<!-- CELL 1 -->
# User Anomaly Analysis Using Graph Builder API

This notebook demonstrates how to build and query a security graph for analyzing user anomalies across multiple Azure/Microsoft 365 resources using the **Graph Builder API**.

## Key Concepts:
1. **GraphSpecBuilder** - Fluent API for building graph specifications
2. **Nodes** - Entities like users, applications, IP addresses, locations
3. **Edges** - Relationships like sign-ins, accesses, connections
4. **Graph Queries** - GQL queries to analyze patterns and anomalies
5. **Visualization** - Interactive graph exploration

## Analysis Goals:
- Multi-resource user activities across different applications
- Anomalous behavior patterns (unusual access times, locations, volumes)
- Cross-platform suspicious activities
- Risk indicators and attack campaign correlations

In [5]:
# CELL 2: Import required libraries and setup
from sentinel_graph.builders import GraphSpecBuilder
from sentinel_graph.core.context import ExecutionContext
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Create execution context
context = ExecutionContext.default()

# Configuration
WORKSPACE_ID = "029c55c8-a7ec-418e-b741-de9d24add5fa"
WORKSPACE_NAME = "Woodgrove-LogAnalyiticsWorkspace"

print("📊 Graph Builder API - User Anomaly Analysis")
print("="*60)
print(f"Workspace: {WORKSPACE_NAME}")
print(f"Analysis Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Context: {context}")
print("="*60)

StatementMeta(MSGGraphInt, 84, 6, Finished, Available, Finished)

2026-03-08 06:46:58 - sentinel_graph.core.context - INFO - ✅ Generated unique graph session ID: kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418


📊 Graph Builder API - User Anomaly Analysis
Workspace: Woodgrove-LogAnalyiticsWorkspace
Analysis Time: 2026-03-08 06:46:58
Context: ExecutionContext(spark=<pyspark.sql.session.SparkSession object at 0x72c77561c880>, step_cache={}, config={}, start_time=None, end_time=None, sentinel_provider=<sentinel_lake.providers.MicrosoftSentinelProvider object at 0x72c77561c760>, default_database='System Tables', database_resolution='default_first', sink_database=None, graph_name='kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418', graph_service_config=None)


<!-- CELL 3 -->
## Step 2: Build Complete Graph Specification

Define all nodes and edges in a single chained call to create the complete graph specification.

In [6]:
# CELL 4: Build complete graph specification
print("🏗️  Building Complete Graph Specification...")
print("="*60)
print("\n📋 This will define:")
print("   • 4 Nodes: user, application, ipaddress, location")
print("   • 3 Edges: signed_in_to, used_ip, from_location")
print()

# Build complete graph in one chain - nodes + edges
graph_spec = (
    GraphSpecBuilder.start(context=context)
    
    # Node 1: User
    .add_node("user")
    .from_table("SigninLogs")
    .with_time_range(time_column="TimeGenerated", lookback_hours=168)
    .with_label("user")
    .with_columns(
        "UserPrincipalName", "UserDisplayName", "UserId",
        "TimeGenerated", "ResultType", "ResultDescription",
        key="UserPrincipalName", display="UserDisplayName"
    )
    
    # Node 2: Application
    .add_node("application")
    .from_table("SigninLogs")
    .with_time_range(time_column="TimeGenerated", lookback_hours=168)
    .with_label("application")
    .with_columns(
        "AppDisplayName", "AppId", "ResourceDisplayName", "TimeGenerated",
        key="AppDisplayName", display="AppDisplayName"
    )
    
    # Node 3: IP Address
    .add_node("ipaddress")
    .from_table("SigninLogs")
    .with_time_range(time_column="TimeGenerated", lookback_hours=168)
    .with_label("ipaddress")
    .with_columns(
        "IPAddress", "TimeGenerated",
        key="IPAddress", display="IPAddress"
    )
    
    # Node 4: Location
    .add_node("location")
    .from_table("SigninLogs")
    .with_time_range(time_column="TimeGenerated", lookback_hours=168)
    .with_label("location")
    .with_columns(
        "Location", "TimeGenerated",
        key="Location", display="Location"
    )
    
    # Edge 1: User → Application
    .add_edge("signed_in")
    .from_table("SigninLogs")
    .with_time_range(time_column="TimeGenerated", lookback_hours=168)
    .edge_label("signed_in_to")
    .source(id_column="UserPrincipalName", node_type="user")
    .target(id_column="AppDisplayName", node_type="application")
    .with_columns(
        "CorrelationId", "TimeGenerated", "ResultType",
        "ResultDescription", "ClientAppUsed",
        key="CorrelationId", display="ResultDescription"
    )
    
    # Edge 2: User → IP Address
    .add_edge("used_ip")
    .from_table("SigninLogs")
    .with_time_range(time_column="TimeGenerated", lookback_hours=168)
    .edge_label("used_ip")
    .source(id_column="UserPrincipalName", node_type="user")
    .target(id_column="IPAddress", node_type="ipaddress")
    .with_columns(
        "CorrelationId", "TimeGenerated", "ResultType",
        key="CorrelationId", display="TimeGenerated"
    )
    
    # Edge 3: User → Location
    .add_edge("from_location")
    .from_table("SigninLogs")
    .with_time_range(time_column="TimeGenerated", lookback_hours=168)
    .edge_label("from_location")
    .source(id_column="UserPrincipalName", node_type="user")
    .target(id_column="Location", node_type="location")
    .with_columns(
        "CorrelationId", "TimeGenerated", "ResultType",
        key="CorrelationId", display="TimeGenerated"
    )
    
    .done()
)

print("✅ Graph Specification Complete!")
print("="*60)
print(f"Graph Name: {graph_spec.name}")
print(f"\nNodes: {len(graph_spec.get_schema().nodes)}")
for node in graph_spec.get_schema().nodes:
    print(f"   • {node.get_primary_label()}")
print(f"\nEdges: {len(graph_spec.get_schema().edges)}")
for edge in graph_spec.get_schema().edges:
    print(f"   • {edge.source_node_label} → {edge.relationship_type} → {edge.target_node_label}")
print("="*60)

StatementMeta(MSGGraphInt, 84, 7, Finished, Available, Finished)

🏗️  Building Complete Graph Specification...

📋 This will define:
   • 4 Nodes: user, application, ipaddress, location
   • 3 Edges: signed_in_to, used_ip, from_location



2026-03-08 06:47:12 - sentinel_graph.catalog.table_resolver - INFO - ✅ Found 'SigninLogs' in database 'CyberSOC'
2026-03-08 06:47:12 - sentinel_graph.builders.base - INFO - Date filtering enabled: lookback 168 hours (2026-03-01 06:47:12.861811 to 2026-03-08 06:47:12.861811)
2026-03-08 06:47:14 - sentinel_graph.catalog.table_resolver - INFO - ✅ Found 'SigninLogs' in database 'CyberSOC'
2026-03-08 06:47:14 - sentinel_graph.builders.base - INFO - Date filtering enabled: lookback 168 hours (2026-03-01 06:47:14.299826 to 2026-03-08 06:47:14.299826)
2026-03-08 06:47:15 - sentinel_graph.catalog.table_resolver - INFO - ✅ Found 'SigninLogs' in database 'CyberSOC'
2026-03-08 06:47:15 - sentinel_graph.builders.base - INFO - Date filtering enabled: lookback 168 hours (2026-03-01 06:47:15.679176 to 2026-03-08 06:47:15.679176)
2026-03-08 06:47:17 - sentinel_graph.catalog.table_resolver - INFO - ✅ Found 'SigninLogs' in database 'CyberSOC'
2026-03-08 06:47:17 - sentinel_graph.builders.base - INFO - Da

✅ Graph Specification Complete!
Graph Name: kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418

Nodes: 4
   • user
   • application
   • ipaddress
   • location

Edges: 3
   • user → signed_in_to → application
   • user → used_ip → ipaddress
   • user → from_location → location


<!-- CELL 5 -->
## Step 3: Build Graph with Data

Execute the ETL pipeline and create the graph instance.

In [7]:
# CELL 6: Build graph with data
print("🚀 Building Graph with Data...")
print("="*60)

# Build the graph with data
result = graph_spec.build_graph_with_data()

print(f"✅ Build Status: {result['status']}")
print(f"Instance Name: {result.get('instance_name', 'N/A')}")
print()

if 'etl_result' in result:
    print("ETL Pipeline Results:")
    print(f"  • Status: Success")
    print(f"  • Result: {result['etl_result']}")

if 'api_result' in result:
    print("API Results:")
    print(f"  • Graph instance created successfully")
    
print("="*60)
print("Graph is ready for querying!")

StatementMeta(MSGGraphInt, 84, 8, Submitted, Running, Running)

🚀 Building Graph with Data...
{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Loading table: SigninLogs"}


2026-03-08 06:48:30 - sentinel_graph.etl.steps - INFO - Applied date filtering to step 'user_table_source' using column 'TimeGenerated' (start: 2026-03-01 06:47:12.861811, end: 2026-03-08 06:47:12.861811)


{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Successfully loaded table SigninLogs"}
{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Loading table: SigninLogs"}


2026-03-08 06:48:38 - sentinel_graph.etl.steps - INFO - Applied date filtering to step 'application_table_source' using column 'TimeGenerated' (start: 2026-03-01 06:47:14.299826, end: 2026-03-08 06:47:14.299826)


...

{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Successfully loaded table SigninLogs"}
{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Loading table: SigninLogs"}


2026-03-08 06:49:18 - sentinel_graph.etl.steps - INFO - Applied date filtering to step 'from_location_table_source_428ca710' using column 'TimeGenerated' (start: 2026-03-01 06:47:20.938226, end: 2026-03-08 06:47:20.938226)


{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Successfully loaded table SigninLogs"}
{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Saving DataFrame as table: graph_kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418_nodes_SPRK"}
{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Successfully saved DataFrame to table graph_kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418_nodes_SPRK with options: {'mode': 'overwrite', 'format': 'delta', 'mergeSchema': 'true', 'overwriteSchema': 'false'}"}
{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Saving DataFrame as table: graph_kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418_edges_SPRK"}
{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Successfully saved DataFrame to table graph_kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418_edges_SPRK with options: 

2026-03-08 06:52:26 - sentinel_graph.core.api - INFO - ✅ Graph instance 'kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418' is ready for queries.


{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Sentinel API call completed - Path: /graphs/custom-graph-instances/kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418, Status: 200, Duration: 261ms, ResponseSize: 821 bytes"}
✅ Build Status: success
Instance Name: kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418

ETL Pipeline Results:
  • Status: Success
  • Result: {'user_table_source': GraphDataFrame(untyped, key_columns=('UserPrincipalName',), columns=94), 'user_node': GraphDataFrame(untyped, key_columns=('UserPrincipalName',), columns=9), 'application_table_source': GraphDataFrame(untyped, key_columns=('AppDisplayName',), columns=94), 'application_node': GraphDataFrame(untyped, key_columns=('AppDisplayName',), columns=7), 'ipaddress_table_source': GraphDataFrame(untyped, key_columns=('IPAddress',), columns=94), 'ipaddress_node': GraphDataFrame(untyped, key_columns=('IPAddress',), columns=5), 'location_table_source': GraphDataFrame(untyp

In [8]:
# CELL 7: Display graph visualization
print("📊 Displaying Graph Data...")
print("="*60)

# Query all user relationships (signed_in_to, used_ip, from_location)
viz_query = (
    "MATCH (u:user)-[r]->(n) "
    "RETURN u, r, n "
    "LIMIT 100"
)

viz_result = graph_spec.query(viz_query)

print("\n✅ Graph visualization displayed above")
print("You can now interactively explore:")
print("  • User nodes and their connections")
print("  • Application access patterns")
print("  • IP address usage")
print("  • Geographic distribution")

viz_result.show(format="visual")

StatementMeta(MSGGraphInt, 84, 9, Finished, Available, Finished)

📊 Displaying Graph Data...
{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Sentinel API call started - Path: /graphs/custom-graph-instances/kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418/query, Method: POST"}


2026-03-08 06:52:36 - sentinel_graph.builders.graph_builder - INFO - ✅ Query executed successfully for graph 'kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418'


{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Sentinel API call completed - Path: /graphs/custom-graph-instances/kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418/query, Status: 200, Duration: 1014ms, ResponseSize: 152911 bytes"}

✅ Graph visualization displayed above
You can now interactively explore:
  • User nodes and their connections
  • Application access patterns
  • IP address usage
  • Geographic distribution


Sentinel Graph Visualization

In [14]:
# CELL 8: Inspect graph schema
print("📋 Inspecting Graph Schema...")
print("="*60)

# Get and display the graph schema
schema = graph_spec.get_schema()

print(f"\nGraph Schema: {schema.name}")
print(f"Version: {schema.get_version()}")
print()

print("Nodes:")
for node in schema.nodes:
    print(f"  • {node.get_primary_label()} (alias: {node.alias})")
    print(f"    - Key: {node.get_primary_key_property_name()}")
    print(f"    - Properties: {len(node.properties)}")
    
print("\nEdges:")
for edge in schema.edges:
    print(f"  • {edge.source_node_label} → [{edge.relationship_type}] → {edge.target_node_label}")
    print(f"    - Properties: {len(edge.properties)}")

print("\n" + "="*60)

StatementMeta(MSGGraphInt, 82, 15, Finished, Available, Finished)

📋 Inspecting Graph Schema...

Graph Schema: kernel_459eb775_486e_49b8_b0af_b5791ba6c0fd_ts_1772807835
Version: 1.0

Nodes:
  • user (alias: user)
    - Key: UserPrincipalName
    - Properties: 6
  • application (alias: application)
    - Key: AppDisplayName
    - Properties: 4
  • ipaddress (alias: ipaddress)
    - Key: IPAddress
    - Properties: 2
  • location (alias: location)
    - Key: Location
    - Properties: 2

Edges:
  • user → [signed_in_to] → application
    - Properties: 5
  • user → [used_ip] → ipaddress
    - Properties: 3
  • user → [from_location] → location
    - Properties: 3



In [ ]:
# CELL 9: Query 1 - Failed sign-ins analysis
print("🔍 Query 1: Find Users with Multiple Failed Sign-ins")
print("="*60)

# Aggregated query for tabular view
query1_table = (
    "MATCH (u:user)-[s:signed_in_to]->(a:application) "
    "WHERE s.ResultType <> '0' "
    "RETURN u.UserPrincipalName, u.UserDisplayName, "
    "count(s) AS FailedAttempts, "
    "count(DISTINCT a.AppDisplayName) AS UniqueApps "
    "ORDER BY FailedAttempts DESC "
    "LIMIT 20"
)

# Graph query for visual view — returns full nodes and edges
query1_visual = (
    "MATCH (u:user)-[s:signed_in_to]->(a:application) "
    "WHERE s.ResultType <> '0' "
    "RETURN u, s, a "
    "LIMIT 100"
)

result1_table = graph_spec.query(query1_table)
result1_visual = graph_spec.query(query1_visual)
print("\n✅ Query executed successfully")

# Tabular view for data inspection
print("\n📋 Tabular Results:")
result1_table.show(format="table")

# Visual graph — highlights user→application failure paths
print("\n🔵 Visual Graph — Failed Sign-in Paths (user → application):")
print("   • Indicates possible credential stuffing attacks")
print("   • Multiple apps suggests targeted reconnaissance")
result1_visual.show(format="visual")

print("\n📊 Analysis: Users with high failed authentication rates")

StatementMeta(MSGGraphInt, 84, 10, Finished, Available, Finished)

🔍 Query 1: Find Users with Multiple Failed Sign-ins
{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Sentinel API call started - Path: /graphs/custom-graph-instances/kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418/query, Method: POST"}


2026-03-08 07:04:11 - sentinel_graph.builders.graph_builder - INFO - ✅ Query executed successfully for graph 'kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418'


{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Sentinel API call completed - Path: /graphs/custom-graph-instances/kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418/query, Status: 200, Duration: 1963ms, ResponseSize: 4575 bytes"}
{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Sentinel API call started - Path: /graphs/custom-graph-instances/kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418/query, Method: POST"}


2026-03-08 07:04:12 - sentinel_graph.builders.graph_builder - INFO - ✅ Query executed successfully for graph 'kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418'


{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Sentinel API call completed - Path: /graphs/custom-graph-instances/kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418/query, Status: 200, Duration: 1084ms, ResponseSize: 172339 bytes"}

✅ Query executed successfully

📋 Tabular Results:
+------------------------+-------------------+--------------+----------+
|u.UserPrincipalName     |u.UserDisplayName  |FailedAttempts|UniqueApps|
+------------------------+-------------------+--------------+----------+
|u1414@int.zava-corp.com |Sofija Klobucar    |167           |4         |
|u421@int.zava-corp.com  |Lina Eriksson      |128           |4         |
|u2470@int.zava-corp.com |Lara Mihajlovic    |128           |5         |
|u13758@int.zava-corp.com|Pedro Shumenko     |109           |4         |
|u529@int.zava-corp.com  |Coralie Taylor     |101           |8         |
|u13513@int.zava-corp.com|Leni Bellod        |93            |4         |
|u3547@int.zava-cor

Sentinel Graph Visualization


📊 Analysis: Users with high failed authentication rates


In [ ]:
# CELL 10: Query 2 - Anomalous IP usage
print("🔍 Query 2: Identify Users with Anomalous IP Usage")
print("="*60)

# Aggregated query for tabular view
query2_table = (
    "MATCH (u:user)-[ip:used_ip]->(i:ipaddress) "
    "RETURN u.UserPrincipalName, u.UserDisplayName, "
    "count(DISTINCT i.IPAddress) AS UniqueIPs, "
    "count(ip) AS TotalConnections "
    "ORDER BY UniqueIPs DESC "
    "LIMIT 15"
)

# Graph query for visual view — returns full nodes and edges
query2_visual = (
    "MATCH (u:user)-[ip:used_ip]->(i:ipaddress) "
    "RETURN u, ip, i "
    "LIMIT 100"
)

result2_table = graph_spec.query(query2_table)
result2_visual = graph_spec.query(query2_visual)
print("\n✅ Query executed successfully")

# Tabular view for data inspection
print("\n📋 Tabular Results:")
result2_table.show(format="table")

# Visual graph — reveals the multi-IP fan-out per user
print("\n🔵 Visual Graph — User → IP Address Fan-Out:")
print("   • High IP diversity may indicate:")
print("     - Distributed attack patterns")
print("     - VPN/proxy usage")
print("     - Compromised credentials being used from multiple locations")
result2_visual.show(format="visual")

print("\n📊 Analysis: Users with multiple IP addresses")

StatementMeta(MSGGraphInt, 84, 12, Finished, Available, Finished)

🔍 Query 2: Identify Users with Anomalous IP Usage
{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Sentinel API call started - Path: /graphs/custom-graph-instances/kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418/query, Method: POST"}


2026-03-08 07:19:01 - sentinel_graph.builders.graph_builder - INFO - ✅ Query executed successfully for graph 'kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418'


{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Sentinel API call completed - Path: /graphs/custom-graph-instances/kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418/query, Status: 200, Duration: 2557ms, ResponseSize: 3530 bytes"}
{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Sentinel API call started - Path: /graphs/custom-graph-instances/kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418/query, Method: POST"}


2026-03-08 07:19:02 - sentinel_graph.builders.graph_builder - INFO - ✅ Query executed successfully for graph 'kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418'


{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Sentinel API call completed - Path: /graphs/custom-graph-instances/kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418/query, Status: 200, Duration: 701ms, ResponseSize: 131831 bytes"}

✅ Query executed successfully

📋 Tabular Results:
+--------------------------------------------------+----------------------------+---------+----------------+
|u.UserPrincipalName                               |u.UserDisplayName           |UniqueIPs|TotalConnections|
+--------------------------------------------------+----------------------------+---------+----------------+
|u12812@int.zava-corp.com                          |Lorato Kamandar             |22       |38              |
|u18960@int.zava-corp.com                          |Rozalia Kralj               |14       |91              |
|u366@int.zava-corp.com                            |Srdjan Zafirovski           |11       |31              |
|u13523@int.zava-corp.c

Sentinel Graph Visualization


📊 Analysis: Users with multiple IP addresses


In [ ]:
# CELL 11: Query 3 - Geographic anomaly detection
print("🔍 Query 3: Geographic Anomaly Detection")
print("="*60)

# Aggregated query for tabular view
query3_table = (
    "MATCH (u:user)-[l:from_location]->(loc:location) "
    "RETURN u.UserPrincipalName, u.UserDisplayName, "
    "count(DISTINCT loc.Location) AS UniqueLocations, "
    "count(l) AS TotalActivities "
    "ORDER BY UniqueLocations DESC "
    "LIMIT 15"
)

# Graph query for visual view — returns full nodes and edges
query3_visual = (
    "MATCH (u:user)-[l:from_location]->(loc:location) "
    "RETURN u, l, loc "
    "LIMIT 100"
)

result3_table = graph_spec.query(query3_table)
result3_visual = graph_spec.query(query3_visual)
print("\n✅ Query executed successfully")

# Tabular view for data inspection
print("\n📋 Tabular Results:")
result3_table.show(format="table")

# Visual graph — geographic spread as user→location node graph
print("\n🔵 Visual Graph — User → Location Geographic Spread:")
print("   • Multiple locations in short timeframe suggests:")
print("     - Impossible travel scenarios")
print("     - Credential compromise")
print("     - Need for deeper investigation")
result3_visual.show(format="visual")

print("\n📊 Analysis: Geographic spread of user activities")

StatementMeta(MSGGraphInt, 84, 14, Finished, Available, Finished)

🔍 Query 3: Geographic Anomaly Detection
{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Sentinel API call started - Path: /graphs/custom-graph-instances/kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418/query, Method: POST"}


2026-03-08 07:22:57 - sentinel_graph.builders.graph_builder - INFO - ✅ Query executed successfully for graph 'kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418'


{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Sentinel API call completed - Path: /graphs/custom-graph-instances/kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418/query, Status: 200, Duration: 1228ms, ResponseSize: 3484 bytes"}
{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Sentinel API call started - Path: /graphs/custom-graph-instances/kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418/query, Method: POST"}


2026-03-08 07:22:57 - sentinel_graph.builders.graph_builder - INFO - ✅ Query executed successfully for graph 'kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418'


{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Sentinel API call completed - Path: /graphs/custom-graph-instances/kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418/query, Status: 200, Duration: 683ms, ResponseSize: 134209 bytes"}

✅ Query executed successfully

📋 Tabular Results:
+------------------------+-------------------+---------------+---------------+
|u.UserPrincipalName     |u.UserDisplayName  |UniqueLocations|TotalActivities|
+------------------------+-------------------+---------------+---------------+
|elviaa@zava-corp.com    |Elvia Atkins       |5              |299            |
|u1013@int.zava-corp.com |Bazarbay Durdyyew  |4              |72             |
|u11725@int.zava-corp.com|Siti Jovovic       |4              |58             |
|u153@a.alpineskihouse.co|Endre Valbe        |4              |590            |
|u9499@int.zava-corp.com |Matiu Kasesalu     |3              |37             |
|u4066@int.zava-corp.com |Palamee Kodikara   

Sentinel Graph Visualization


📊 Analysis: Geographic spread of user activities


<!-- CELL 12 -->
## Step 4: Advanced Analysis - Multi-Factor Anomaly Detection

Combine multiple signals to identify high-risk users.

In [20]:
# CELL 13: Query 4 - Multi-factor risk analysis
print("🚨 Query 4: Multi-Factor Risk Analysis")
print("="*60)

# Step 1 — Aggregated table query: rank users by failed sign-ins and apps targeted
query4_table = (
    "MATCH (u:user)-[s:signed_in_to]->(a:application) "
    "WHERE s.ResultType <> '0' "
    "RETURN u.UserPrincipalName AS UserPrincipalName, u.UserDisplayName AS UserDisplayName, "
    "count(DISTINCT s) AS FailedSignIns, "
    "count(DISTINCT a.AppDisplayName) AS AppsTargeted "
    "ORDER BY FailedSignIns DESC, AppsTargeted DESC "
    "LIMIT 10"
)

result4_table = graph_spec.query(query4_table)
print("\n✅ Table query executed successfully")

print("\n📋 Risk Ranking Table:")
result4_table.show(format="table")

# Step 2 — Extract top user names from Spark DataFrame and build targeted visual query
try:
    top_df = result4_table.to_dataframe()
    top_rows = top_df.limit(5).collect()
    if top_rows:
        top_users = [row["UserPrincipalName"] for row in top_rows]
        user_filter = ", ".join(f"'{u}'" for u in top_users)

        # Visual query: all connections for the top high-risk users
        query4_visual = (
            "MATCH (u:user)-[r]->(n) "
            f"WHERE u.UserPrincipalName IN [{user_filter}] "
            "RETURN u, r, n "
            "LIMIT 200"
        )
        result4_visual = graph_spec.query(query4_visual)
        print("\n🔴 Visual Graph — Multi-Factor Attack Surface (Top High-Risk Users):")
        print("   • High failed sign-ins + Multiple apps = Credential stuffing")
        print("   • Multiple IPs + Multiple locations = Distributed attack")
        print("   • All factors combined = Critical threat - Immediate action required")
        result4_visual.show(format="visual")
    else:
        print("⚠️  No high-risk users found in table results")
except Exception as e:
    print(f"⚠️  Visual query failed ({e})")

print("\n🔥 HIGH-RISK INDICATORS:")
print("   • Users appearing in all three risk categories require immediate investigation")

StatementMeta(MSGGraphInt, 84, 21, Finished, Available, Finished)

🚨 Query 4: Multi-Factor Risk Analysis
{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Sentinel API call started - Path: /graphs/custom-graph-instances/kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418/query, Method: POST"}


2026-03-08 07:28:46 - sentinel_graph.builders.graph_builder - INFO - ✅ Query executed successfully for graph 'kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418'


{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Sentinel API call completed - Path: /graphs/custom-graph-instances/kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418/query, Status: 200, Duration: 1068ms, ResponseSize: 2364 bytes"}

✅ Table query executed successfully

📋 Risk Ranking Table:
+------------------------+---------------+-------------+------------+
|UserPrincipalName       |UserDisplayName|FailedSignIns|AppsTargeted|
+------------------------+---------------+-------------+------------+
|u1414@int.zava-corp.com |Sofija Klobucar|167          |4           |
|u2470@int.zava-corp.com |Lara Mihajlovic|128          |5           |
|u421@int.zava-corp.com  |Lina Eriksson  |128          |4           |
|u13758@int.zava-corp.com|Pedro Shumenko |109          |4           |
|u529@int.zava-corp.com  |Coralie Taylor |101          |8           |
|u13513@int.zava-corp.com|Leni Bellod    |93           |4           |
|u3547@int.zava-corp.com |Jawid Jaunzem

2026-03-08 07:28:48 - sentinel_graph.builders.graph_builder - INFO - ✅ Query executed successfully for graph 'kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418'


{"Level": "INFO", "TraceId": "e53c917c-9a1d-4306-934f-1b8f1a0d7a20", "Message": "Sentinel API call completed - Path: /graphs/custom-graph-instances/kernel_c9333a44_fee5_4fbc_abb9_d6d0030f16fe_ts_1772952418/query, Status: 200, Duration: 880ms, ResponseSize: 282238 bytes"}

🔴 Visual Graph — Multi-Factor Attack Surface (Top High-Risk Users):
   • High failed sign-ins + Multiple apps = Credential stuffing
   • Multiple IPs + Multiple locations = Distributed attack
   • All factors combined = Critical threat - Immediate action required


Sentinel Graph Visualization


🔥 HIGH-RISK INDICATORS:
   • Users appearing in all three risk categories require immediate investigation


<!-- CELL 14 -->
## Step 5: Graph Algorithms with GraphFrame

Convert the graph to GraphFrame format to run advanced graph algorithms like PageRank, Connected Components, etc.

In [4]:
# CELL 15: GraphFrame algorithms (PageRank)
print("📈 Running Graph Algorithms with GraphFrame")
print("="*60)

# Convert graph to GraphFrame
gf = graph_spec.to_graphframe()

print("✅ GraphFrame created")
print(f"Vertices: {gf.vertices.count():,}")
print(f"Edges: {gf.edges.count():,}")
print()

# Run PageRank to identify influential nodes
print("Running PageRank algorithm...")
pagerank_result = gf.pageRank(resetProbability=0.15, maxIter=10)

print("\n📊 Top 20 Most Influential Nodes (by PageRank):")
print("-"*60)
pagerank_result.vertices \
    .select("id", "pagerank") \
    .orderBy("pagerank", ascending=False) \
    .limit(20) \
    .show(truncate=False)

print("\n💡 Interpretation:")
print("   • High PageRank = Central nodes in the attack graph")
print("   • These users/IPs/apps are highly connected")
print("   • Priority targets for investigation and remediation")
print("="*60)

StatementMeta(MSGGraphInt, 85, 5, Finished, Available, Finished)

📈 Running Graph Algorithms with GraphFrame


NameError: name 'graph_spec' is not defined